In [ ]:
import sys
print(sys.executable)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

SAVE_DIR = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\data\processed'

# Load
X_train = np.load(f'{SAVE_DIR}\\X_train.npy')
X_dev   = np.load(f'{SAVE_DIR}\\X_dev.npy')
X_eval  = np.load(f'{SAVE_DIR}\\X_eval.npy')
y_train = np.load(f'{SAVE_DIR}\\y_train.npy')
y_dev   = np.load(f'{SAVE_DIR}\\y_dev.npy')
y_eval  = np.load(f'{SAVE_DIR}\\y_eval.npy')

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_dev_sc   = scaler.transform(X_dev)
X_eval_sc  = scaler.transform(X_eval)

print(f"X_train: {X_train_sc.shape}")
print(f"X_dev:   {X_dev_sc.shape}")
print(f"X_eval:  {X_eval_sc.shape}")
print(f"\nTrain — Spoof: {y_train.sum()}, Bonafide: {(y_train==0).sum()}")
print("Scaling done ✅")

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_sc, y_train)

print(f"Before SMOTE — Spoof: {y_train.sum()}, Bonafide: {(y_train==0).sum()}")
print(f"After SMOTE  — Spoof: {y_train_bal.sum()}, Bonafide: {(y_train_bal==0).sum()}")
print(f"New shape: {X_train_bal.shape}")

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import time

# XGBoost train
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss',
    device='cuda'  # RTX 4050 use hoga
)

start = time.time()
xgb.fit(X_train_bal, y_train_bal)
end = time.time()

print(f"Training time: {(end-start):.1f}s")

# Dev pe evaluate karo
y_pred = xgb.predict(X_dev_sc)
print("\n=== XGBoost — Dev Set ===")
print(classification_report(y_dev, y_pred, target_names=['bonafide', 'spoof']))

In [ ]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    device='gpu'
)

start = time.time()
lgbm.fit(X_train_bal, y_train_bal)
end = time.time()

print(f"Training time: {(end-start):.1f}s")

y_pred_lgbm = lgbm.predict(X_dev_sc)
print("\n=== LightGBM — Dev Set ===")
print(classification_report(y_dev, y_pred_lgbm, target_names=['bonafide', 'spoof']))

In [ ]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=100,
    depth=6,
    learning_rate=0.1,
    random_seed=42,
    task_type='GPU',
    verbose=0
)

start = time.time()
cat.fit(X_train_bal, y_train_bal)
end = time.time()

print(f"Training time: {(end-start):.1f}s")

y_pred_cat = cat.predict(X_dev_sc)
print("\n=== CatBoost — Dev Set ===")
print(classification_report(y_dev, y_pred_cat, target_names=['bonafide', 'spoof']))

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Base learners
estimators = [
    ('xgb', XGBClassifier(n_estimators=100, max_depth=6, 
                           learning_rate=0.1, random_state=42,
                           eval_metric='logloss')),
    ('lgbm', LGBMClassifier(n_estimators=100, max_depth=6,
                             learning_rate=0.1, random_state=42,
                             verbose=-1)),
    ('cat', CatBoostClassifier(iterations=100, depth=6,
                                learning_rate=0.1, random_seed=42,
                                verbose=0))
]

# Meta learner
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

start = time.time()
stack.fit(X_train_bal, y_train_bal)
end = time.time()

print(f"Training time: {(end-start):.1f}s")

y_pred_stack = stack.predict(X_dev_sc)
print("\n=== Stacking Ensemble — Dev Set ===")
print(classification_report(y_dev, y_pred_stack, 
      target_names=['bonafide', 'spoof']))

In [ ]:
import pandas as pd

results = {
    'Model': ['XGBoost', 'LightGBM', 'CatBoost', 'Stacking Ensemble'],
    'Accuracy': [0.99, 0.99, 0.99, 0.99],
    'Bonafide Recall': [0.95, 0.95, 0.97, 0.88],
    'Spoof Recall': [0.99, 0.99, 0.99, 1.00],
    'Macro F1': [0.97, 0.97, 0.97, 0.96],
    'Train Time (s)': [1.0, 1.5, 2.0, 32.4]
}

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Save
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
table = ax.table(cellText=df_results.values,
                 colLabels=df_results.columns,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
plt.tight_layout()
plt.savefig(r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\figures\model_comparison.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved ✅")

In [ ]:
print("=== EVAL SET — Final Test ===\n")

for name, pred in [
    ("XGBoost",  xgb.predict(X_eval_sc)),
    ("LightGBM", lgbm.predict(X_eval_sc)),
    ("CatBoost", cat.predict(X_eval_sc)),
]:
    print(f"--- {name} ---")
    print(classification_report(y_eval, pred, 
          target_names=['bonafide', 'spoof']))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['XGBoost', 'LightGBM', 'CatBoost']

dev_spoof_recall  = [0.99, 0.99, 0.99]
eval_spoof_recall = [0.70, 0.70, 0.70]

dev_bonafide_recall  = [0.95, 0.95, 0.97]
eval_bonafide_recall = [0.96, 0.97, 0.96]

x = np.arange(len(models))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Spoof recall
axes[0].bar(x - width/2, dev_spoof_recall,  width, label='Dev (known attacks)',   color='green',  alpha=0.8)
axes[0].bar(x + width/2, eval_spoof_recall, width, label='Eval (unknown attacks)', color='red', alpha=0.8)
axes[0].set_title('Spoof Recall — Known vs Unknown Attacks', fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Recall')
axes[0].legend()
axes[0].axhline(y=0.99, color='green', linestyle='--', alpha=0.5)
axes[0].axhline(y=0.70, color='red',   linestyle='--', alpha=0.5)

for i, (d, e) in enumerate(zip(dev_spoof_recall, eval_spoof_recall)):
    axes[0].text(i - width/2, d + 0.01, f'{d:.2f}', ha='center', fontsize=10)
    axes[0].text(i + width/2, e + 0.01, f'{e:.2f}', ha='center', fontsize=10)

# Bonafide recall
axes[1].bar(x - width/2, dev_bonafide_recall,  width, label='Dev (known attacks)',   color='green', alpha=0.8)
axes[1].bar(x + width/2, eval_bonafide_recall, width, label='Eval (unknown attacks)', color='red', alpha=0.8)
axes[1].set_title('Bonafide Recall — Known vs Unknown Attacks', fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Recall')
axes[1].legend()

for i, (d, e) in enumerate(zip(dev_bonafide_recall, eval_bonafide_recall)):
    axes[1].text(i - width/2, d + 0.01, f'{d:.2f}', ha='center', fontsize=10)
    axes[1].text(i + width/2, e + 0.01, f'{e:.2f}', ha='center', fontsize=10)

plt.suptitle('Key Finding: Models Fail on Unknown Attack Types', 
             fontsize=15, fontweight='bold', color='red')
plt.tight_layout()
plt.savefig(r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\figures\known_vs_unknown.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved ✅")

In [ ]:
import joblib

MODELS_DIR = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results'

joblib.dump(xgb,    f'{MODELS_DIR}\\xgboost_model.pkl')
joblib.dump(lgbm,   f'{MODELS_DIR}\\lightgbm_model.pkl')
joblib.dump(cat,    f'{MODELS_DIR}\\catboost_model.pkl')
joblib.dump(scaler, f'{MODELS_DIR}\\scaler.pkl')

print("Models saved ✅")
print(f"XGBoost:  xgboost_model.pkl")
print(f"LightGBM: lightgbm_model.pkl")
print(f"CatBoost: catboost_model.pkl")
print(f"Scaler:   scaler.pkl")